In [ ]:
import numpy as np 
import matplotlib.pyplot as plt 
from astropy.io import fits
from astropy.table import Table
from astropy.table import vstack
import os

from scipy import stats
import tensorflow as tf
import tensorflow_datasets as tfds
import keras_tuner as kt
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns

### Import data

In [ ]:
fits_file_2384a = '/work/mccleary_group/saha/data/Abell2384a/sextractor_dualmode/out/Abell2384a_colors_mags.fits'
hdul_2384a = fits.open(fits_file_2384a)
data_2384a = Table(hdul_2384a[1].data)

fits_file_2384b = '/work/mccleary_group/saha/data/Abell2384b/sextractor_dualmode/out/Abell2384b_colors_mags.fits'
hdul_2384b = fits.open(fits_file_2384b)
data_2384b = Table(hdul_2384b[1].data)

fits_file_3667 = '/work/mccleary_group/saha/data/Abell3667/sextractor_dualmode/out/Abell3667_colors_mags.fits'
hdul_3667 = fits.open(fits_file_3667)
data_3667 = Table(hdul_3667[1].data)

fits_file_3571 = '/work/mccleary_group/saha/data/Abell3571/sextractor_dualmode/out/Abell3571_colors_mags.fits'
hdul_3571 = fits.open(fits_file_3571)
data_3571 = Table(hdul_3571[1].data)

fits_file_3827 = '/work/mccleary_group/saha/data/Abell3827/sextractor_dualmode/out/Abell3827_colors_mags.fits'
hdul_3827 = fits.open(fits_file_3827)
data_3827 = Table(hdul_3827[1].data)

data = vstack([data_2384a, data_2384b, data_3667, data_3571, data_3827])


data_z = data[~np.isnan(np.array(data['redshift']))]
x_cols = ['m_b', 'm_g', 'm_u', 'R_b', 'R_g',  'R_u']

x = np.array([data_z[x_cols[i]] for i in range(len(x_cols))]).T
y = np.array([data_z['redshift']]).T

### Import model

In [ ]:
path = os.getcwd() + '/mlp_redshift.keras'

check_model = tf.keras.models.load_model(path)

### Compare true and predicted redshifts

In [ ]:
model_out = check_model.predict(x)

plot_bool = (model_out > 0 ) & (model_out < 5)

ob_excluded = len(np.where(~plot_bool)[0])

plt.scatter(y, model_out, c='r', label = f'Objects Excluded: {ob_excluded}', s=5)

mod_min = np.min(model_out[plot_bool])
mod_max = np.max(model_out[plot_bool])
line = np.linspace(mod_min - 5, mod_max + 5, 100)
plt.plot(line, line, c='k')

plt.xlim(-0.1, 2)
plt.ylim(-0.1, 2)

plt.title('Training Dataset')
plt.xlabel('BPZ Redshifts', fontsize = 15)
plt.ylabel('MLP Redshifts', fontsize = 15)

plt.legend()

# CNN

### Visualize Vignettes

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4), gridspec_kw={'wspace': 0.1})

obj = 3

upp = 0.99
low = 0.5

# Plot into each subplot
axs[0].imshow(data_z['VIGNET_u'][obj], vmin=np.nanquantile(data_z['VIGNET_u'][obj], low), 
vmax=np.nanquantile(data_z['VIGNET_u'][obj], upp))

axs[1].imshow(data_z['VIGNET_b'][obj], vmin=np.nanquantile(data_z['VIGNET_b'][obj], low), 
vmax=np.nanquantile(data_z['VIGNET_b'][obj], upp))

axs[2].imshow(data_z['VIGNET_g'][obj], vmin=np.nanquantile(data_z['VIGNET_g'][obj], low), 
vmax=np.nanquantile(data_z['VIGNET_g'][obj], upp))

### Import CNN model

In [ ]:
path = os.getcwd() + '/cnn_redshift.keras'

cnn_model = tf.keras.models.load_model(path)

### CNN test data and predictions

In [ ]:
data_path = os.getcwd().replace('ml_redshifts', 'data') + '/cnn_data/'

trainx = np.load(data_path + 'trainx_cnn.npy')
trainy = np.load(data_path + 'trainy_cnn.npy')
testx = np.load(data_path + 'testx_cnn.npy')
testy = np.load(data_path + 'testy_cnn.npy')

### CNN predictions

In [ ]:
cnn_pred_train = cnn_model.predict(trainx)
cnn_pred_test = cnn_model.predict(testx)

### CNN comparison

In [ ]:
plt.scatter(trainy, cnn_pred_train, s=5, c='b', label='Training Data')
plt.scatter(testy, cnn_pred_test, s=5, c='r', label='Test Data')

one_to_one = np.linspace(0, 2, 100)
plt.plot(one_to_one, one_to_one, c='k')

plt.xlim(0,2)
plt.ylim(0,2)

plt.xlabel('BPZ Redshifts', fontsize = 15)
plt.ylabel('CNN Vignette predictions', fontsize = 15)
plt.legend()